In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  04_streamlit.ipynb — Preparação e deploy do app          ║
# ║  TCC: Cobertura Vacinal no Piauí (2015–2022)               ║
# ╚══════════════════════════════════════════════════════════════╝
#
# Este notebook:
#   1. Copia os parquets para a pasta data/ do app
#   2. Testa o app localmente via ngrok
#   3. Guia o deploy no Streamlit Community Cloud

In [2]:
# ── CÉLULA 1: Setup ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

RAIZ           = '/content/drive/MyDrive/TCC_Vacinal_Piaui'
DADOS_TRATADOS = f'{RAIZ}/dados_tratados'
APP_DIR        = f'{RAIZ}/streamlit_app'
DATA_DIR       = f'{APP_DIR}/data'

os.makedirs(DATA_DIR, exist_ok=True)
print('✅ Pastas prontas')
print(f'   App:  {APP_DIR}')
print(f'   Data: {DATA_DIR}')

Mounted at /content/drive
✅ Pastas prontas
   App:  /content/drive/MyDrive/TCC_Vacinal_Piaui/streamlit_app
   Data: /content/drive/MyDrive/TCC_Vacinal_Piaui/streamlit_app/data


In [3]:
# ── CÉLULA 2: Copiar arquivos necessários para o app ──────────

arquivos = {
    f'{DADOS_TRATADOS}/pni_piaui_unificado.parquet': f'{DATA_DIR}/pni_piaui_unificado.parquet',
    f'{DADOS_TRATADOS}/pni_piaui_clean.parquet':     f'{DATA_DIR}/pni_piaui_clean.parquet',
}

print('📦 Copiando dados para streamlit_app/data/...')
for origem, destino in arquivos.items():
    if os.path.exists(origem):
        shutil.copy2(origem, destino)
        tam = os.path.getsize(destino) / 1024
        print(f'  ✅ {os.path.basename(destino)} ({tam:.0f} KB)')
    else:
        print(f'  ❌ {origem} — NÃO ENCONTRADO')
        print('     Execute o 02_limpeza.ipynb primeiro.')

print()
print('Conteúdo de streamlit_app/:')
for f in os.listdir(APP_DIR):
    print(f'  {f}')

📦 Copiando dados para streamlit_app/data/...
  ✅ pni_piaui_unificado.parquet (43 KB)
  ✅ pni_piaui_clean.parquet (3807 KB)

Conteúdo de streamlit_app/:
  app.py
  requirements.txt
  data


In [4]:
# ── CÉLULA 3: Instalar dependências e testar app localmente ───
# Usa pyngrok para expor o app com URL pública temporária

!pip install streamlit pyngrok kaleido --quiet

# Verificar que o app.py está no lugar certo
app_py = f'{APP_DIR}/app.py'
if not os.path.exists(app_py):
    print('❌ app.py não encontrado em streamlit_app/')
    print('   Faça upload do app.py para o Drive em:')
    print(f'   {APP_DIR}/app.py')
else:
    print(f'✅ app.py encontrado ({os.path.getsize(app_py)/1024:.0f} KB)')

req_txt = f'{APP_DIR}/requirements.txt'
if os.path.exists(req_txt):
    print(f'✅ requirements.txt encontrado')
else:
    print('⚠️  requirements.txt não encontrado — crie antes do deploy')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 57.9 MB/s eta 0:00:00
✅ app.py encontrado (35 KB)
✅ requirements.txt encontrado


In [5]:
# ── CÉLULA 4: Rodar o app com ngrok ───────────────────────────
# ⚠️  Substitua YOUR_NGROK_TOKEN pelo seu token em https://ngrok.com
#     Conta gratuita é suficiente para testar

NGROK_TOKEN = 'YOUR_NGROK_TOKEN'   # ← substitua aqui

if NGROK_TOKEN == 'YOUR_NGROK_TOKEN':
    print('⚠️  Configure seu token do ngrok antes de rodar esta célula.')
    print('   1. Acesse https://ngrok.com e crie uma conta gratuita')
    print('   2. Copie seu token em https://dashboard.ngrok.com/get-started/your-authtoken')
    print('   3. Substitua YOUR_NGROK_TOKEN acima e rode novamente')
else:
    from pyngrok import ngrok, conf
    import subprocess, threading, time

    conf.get_default().auth_token = NGROK_TOKEN

    # Iniciar Streamlit em background
    proc = subprocess.Popen(
        ['streamlit', 'run', app_py,
         '--server.port', '8501',
         '--server.headless', 'true'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )

    time.sleep(5)  # Aguardar o servidor subir

    # Expor com ngrok
    tunnel = ngrok.connect(8501)
    print('='*55)
    print('✅ App rodando! Acesse:')
    print(f'   {tunnel.public_url}')
    print('='*55)
    print('⚠️  O link é temporário e expira quando esta célula parar.')
    print('   Para parar: Ambiente de execução > Interromper execução')

⚠️  Configure seu token do ngrok antes de rodar esta célula.
   1. Acesse https://ngrok.com e crie uma conta gratuita
   2. Copie seu token em https://dashboard.ngrok.com/get-started/your-authtoken
   3. Substitua YOUR_NGROK_TOKEN acima e rode novamente


In [6]:
# ── CÉLULA 5: Preparar repositório GitHub para deploy ─────────
# O Streamlit Community Cloud faz deploy direto do GitHub

print("""
╔══════════════════════════════════════════════════════════════╗
║  GUIA DE DEPLOY — Streamlit Community Cloud                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  ESTRUTURA DO REPOSITÓRIO GITHUB:                            ║
║                                                              ║
║  tcc-vacinal-piaui/                                          ║
║  ├── app.py                  ← arquivo principal            ║
║  ├── requirements.txt        ← dependências                 ║
║  └── data/                                                   ║
║      ├── pni_piaui_unificado.parquet                        ║
║      └── pni_piaui_clean.parquet                            ║
║                                                              ║
║  PASSOS:                                                     ║
║  1. Crie um repositório no GitHub (público ou privado)      ║
║  2. Faça upload dos arquivos acima                          ║
║  3. Acesse https://share.streamlit.io                       ║
║  4. Clique em 'New app'                                     ║
║  5. Conecte ao repositório GitHub                           ║
║  6. Main file path: app.py                                  ║
║  7. Clique 'Deploy!' — URL pública em ~2 minutos            ║
║                                                              ║
║  DICA: O deploy é GRATUITO para repositórios públicos.      ║
║  Para repositório privado, use conta Streamlit gratuita.    ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║  GUIA DE DEPLOY — Streamlit Community Cloud                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  ESTRUTURA DO REPOSITÓRIO GITHUB:                            ║
║                                                              ║
║  tcc-vacinal-piaui/                                          ║
║  ├── app.py                  ← arquivo principal            ║
║  ├── requirements.txt        ← dependências                 ║
║  └── data/                                                   ║
║      ├── pni_piaui_unificado.parquet                        ║
║      └── pni_piaui_clean.parquet                            ║
║                                                              ║
║  PASSOS:                                                     ║
║  1. Crie um repositório no GitHub (público ou privado)      ║
║  2. Faça upload dos arquivos

In [ ]:
# ── CÉLULA 6: Upload automático para GitHub via API ───────────
# Opcional — só use se quiser automatizar o upload
# Requer Personal Access Token do GitHub com permissão 'repo'

import base64, requests as req

GITHUB_TOKEN = 'YOUR_GITHUB_TOKEN'   # ← substitua
GITHUB_USER  = 'seu_usuario'          # ← substitua
GITHUB_REPO  = 'tcc-vacinal-piaui'   # ← substitua (crie o repo antes)

def upload_github(caminho_local, caminho_repo, token, user, repo):
    """Faz upload de um arquivo para o GitHub via API."""
    with open(caminho_local, 'rb') as f:
        conteudo = base64.b64encode(f.read()).decode()

    url = f'https://api.github.com/repos/{user}/{repo}/contents/{caminho_repo}'
    headers = {
        'Authorization': f'token {token}',
        'Accept': 'application/vnd.github.v3+json'
    }

    # Verificar se arquivo já existe (para update)
    r = req.get(url, headers=headers)
    sha = r.json().get('sha') if r.status_code == 200 else None

    payload = {
        'message': f'Upload {os.path.basename(caminho_repo)}',
        'content': conteudo,
    }
    if sha:
        payload['sha'] = sha

    r = req.put(url, headers=headers, json=payload)
    return r.status_code in [200, 201]


if GITHUB_TOKEN == 'YOUR_GITHUB_TOKEN':
    print('⚠️  Configure GITHUB_TOKEN, GITHUB_USER e GITHUB_REPO para usar esta célula.')
    print('   Obtenha um token em: GitHub → Settings → Developer settings → Personal access tokens')
else:
    arquivos_github = [
        (f'{APP_DIR}/app.py',           'app.py'),
        (f'{APP_DIR}/requirements.txt', 'requirements.txt'),
        (f'{DATA_DIR}/pni_piaui_unificado.parquet', 'data/pni_piaui_unificado.parquet'),
        (f'{DATA_DIR}/pni_piaui_clean.parquet',     'data/pni_piaui_clean.parquet'),
    ]

    print('📤 Enviando para GitHub...')
    for local, repo_path in arquivos_github:
        if os.path.exists(local):
            ok = upload_github(local, repo_path, GITHUB_TOKEN, GITHUB_USER, GITHUB_REPO)
            print(f'  {"✅" if ok else "❌"} {repo_path}')
        else:
            print(f'  ⚠️  {local} não encontrado')

    print(f'\n🚀 Acesse: https://share.streamlit.io para fazer o deploy!')